# Week 3, day 2 — Worksheet 09 SOLUTIONS: outliers   (L05)

Executed in the lab image (pandas 3.0.5) against the real
`data/orders_long.csv`. Every quoted number is what it actually printed.

Questions 4 and 7 are the ones to re-read. The two methods flag very different
numbers of rows, and removing the flagged ones changes the total by real money.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 09 — Outliers. Run this once.
import pandas as pd

orders = pd.read_csv("data/orders_long.csv")
sales = orders["Sales"]

print(sales.describe().round(2).to_string())

PART A — the IQR fence

### Question 1

`Q1 117.78`, `Q3 1618.31`, `IQR 1500.53`, lower fence **`-2133.01`**, upper fence `3869.11`.

The lower fence is negative, which tells you something before you count
anything: this column is bounded below at zero and stretches a long way
above, so the 1.5xIQR rule cannot flag anything at the bottom.

That asymmetry is not a bug in the rule; it is the rule correctly reporting
that a right-skewed column has no small outliers to find.

In [ ]:
q1 = sales.quantile(0.25)
q3 = sales.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print("Q1:    %.2f" % q1)
print("Q3:    %.2f" % q3)
print("IQR:   %.2f" % iqr)
print("lower: %.2f" % lower)
print("upper: %.2f" % upper)

### Question 2

**0** below, **123** above. -> `123 of 1093` — **11.3%** of the file.

Nothing below, as Q1 predicted: the minimum sale is `3.20` and the fence is
at `-2133.01`.

11.3% is a lot to call 'outliers'. The word suggests rare, and one order in
nine is not rare — it is a feature of the distribution. That is the first
sign the rule is describing skew rather than finding anomalies.

In [ ]:
q1, q3 = sales.quantile(0.25), sales.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

below = (sales < lower).sum()
above = (sales > upper).sum()
print("below the lower fence:", below)
print("above the upper fence:", above)
print("total flagged: %d of %d (%.1f%%)"
      % (below + above, len(sales), 100 * (below + above) / len(sales)))
print()
print("minimum sale: %.2f | lower fence: %.2f" % (sales.min(), lower))

### Question 3

123 flagged rows; the largest are ordinary-looking orders across several regions and categories.

This is the step the deck insists on and everyone skips. Nothing in the
flagged rows looks like a data-entry error — no negative quantities, no
impossible dates, no duplicated IDs. They are simply large orders.

'Outlier' is a statement about a value's position in a distribution. It is
not a statement about correctness, and only looking at the rows can tell
you which you have.

In [ ]:
q1, q3 = sales.quantile(0.25), sales.quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
flagged = orders[orders["Sales"] > upper]
print("flagged rows:", len(flagged))
print()
print(flagged.nlargest(10, "Sales")[["OrderID", "Region", "Category", "Sales"]]
      .to_string(index=False))

PART B — the z-score

### Question 4

z-score `|z| > 3` flags **27**; the IQR fence flags **123**. -> a ratio of **4.6x**. Max `|z|` is `10.36`.

Two standard rules, same column, and one finds four and a half times as
many rows as the other.

Neither is wrong. They measure different things: the IQR fence is built
from quartiles, which barely move when the tail gets longer, while the
z-score divides by the standard deviation, which the tail inflates. Q5 and
Q6 show that happening.

The practical point is that 'remove the outliers' is not an instruction. It
names no method, and the two obvious methods disagree by a factor of five.

In [ ]:
z = (sales - sales.mean()) / sales.std()
z_flag = (z.abs() > 3).sum()

q1, q3 = sales.quantile(0.25), sales.quantile(0.75)
iqr = q3 - q1
iqr_flag = ((sales < q1 - 1.5 * iqr) | (sales > q3 + 1.5 * iqr)).sum()

print("z-score |z| > 3 flags:", z_flag)
print("IQR fence flags:      ", iqr_flag)
print("ratio: %.1fx" % (iqr_flag / z_flag))
print()
print("max |z|: %.2f" % z.abs().max())

### Question 5

mean `1468.96`, median **`404.91`**, std `2599.38`, skew **`3.74`**. -> mean is `3.63` times the median.

A skew of 3.74 is strongly right-tailed, and the mean sitting 3.6x above
the median says the same thing in plainer terms: most orders are small and a
few are very large.

The standard deviation is `2599.38` — nearly twice the mean and six times
the median. That number is what the z-score divides by, and the tail is what
made it so big.

So on skewed data the z-score fence is **stretched by the very values it is
supposed to find**. The deck says to use IQR when the data is skewed; this is
the reason.

In [ ]:
print("mean:   %.2f" % sales.mean())
print("median: %.2f" % sales.median())
print("std:    %.2f" % sales.std())
print("skew:   %.2f" % sales.skew())
print()
print("mean / median ratio: %.2f" % (sales.mean() / sales.median()))

# The mean sits far above the median and the skew is strongly positive, so
# the long right tail inflates the standard deviation -- which is the very
# quantity the z-score divides by. On skewed data the z-score fence is
# stretched by the outliers it is meant to find.

### Question 6

std drops from `2599.38` to **`878.63`** when the IQR-flagged rows are removed — a factor of 3. -> within the trimmed data, **18** rows are still `|z| > 3`.

Removing 11% of rows cut the standard deviation to a third. That is the
mechanism from Q5, measured.

The second number is the interesting one. Trim the data and 18 *new* rows
become outliers by the z-score rule, because the yardstick shrank. Do it
again and you would find more.

Outlier removal is not a one-pass operation that converges on 'the clean
data'. Every pass redefines the threshold, and you can keep going until
almost nothing is left. Decide the rule once, apply it once, and do not
iterate.

In [ ]:
q1, q3 = sales.quantile(0.25), sales.quantile(0.75)
iqr = q3 - q1
keep = (sales >= q1 - 1.5 * iqr) & (sales <= q3 + 1.5 * iqr)
trimmed = sales[keep]

print("std, all rows:     %.2f" % sales.std())
print("std, IQR-trimmed:  %.2f" % trimmed.std())
print()
z2 = (trimmed - trimmed.mean()) / trimmed.std()
print("rows in trimmed data:", len(trimmed))
print("|z| > 3 within the trimmed data:", (z2.abs() > 3).sum())

PART C — what removing them costs

### Question 7

All rows: `1093`, total `1605576.22`, mean `1468.96`. Trimmed: `970`, total `693629.62`, mean `715.08`. -> removing **11.3% of rows** removes **56.8% of revenue**.

This is the number to remember from the whole sheet.

One order in nine carries more than half the money. Delete them as
'outliers' and the remaining dataset describes a business with 57% less
revenue and a mean order value of `715.08` instead of `1468.96` — less than
half.

Every one of those 123 orders is real. They are not typos; they are the
large customers. A model trained on the trimmed data has never seen the
transactions that matter most, and a report built on it understates the
business by more than half while containing no errors at all.

The defensible uses of outlier detection are **flagging for review** and
**excluding known-bad records**. 'Trim the tail so the statistics look
nicer' is neither.

In [ ]:
q1, q3 = sales.quantile(0.25), sales.quantile(0.75)
iqr = q3 - q1
keep = (sales >= q1 - 1.5 * iqr) & (sales <= q3 + 1.5 * iqr)

print("            rows      total Sales     mean")
print("all      %6d  %14.2f  %8.2f" % (len(sales), sales.sum(), sales.mean()))
print("trimmed  %6d  %14.2f  %8.2f"
      % (keep.sum(), sales[keep].sum(), sales[keep].mean()))
print()
lost = sales[~keep].sum()
print("revenue removed: %.2f (%.1f%% of the total) in %d orders (%.1f%% of rows)"
      % (lost, 100 * lost / sales.sum(), (~keep).sum(),
         100 * (~keep).sum() / len(sales)))

### Question 8

Per region, flagged and unflagged split out — the flagged mean is several times the unflagged mean in every region.

Marking rather than deleting keeps the decision reversible and lets you ask
the better question: not 'should these be removed' but 'what are they'.

Every region has some. That is itself informative — if the flagged rows had
clustered in one region, you would be looking at a data-collection problem
rather than a distribution.

In [ ]:
work = orders.copy()
q1, q3 = sales.quantile(0.25), sales.quantile(0.75)
iqr = q3 - q1
work["Outlier"] = (work["Sales"] < q1 - 1.5 * iqr) | (work["Sales"] > q3 + 1.5 * iqr)

summary = work.groupby(["Region", "Outlier"]).agg(
    orders=("OrderID", "count"),
    mean_sales=("Sales", "mean"),
).round(2)
print(summary.to_string())

### Question 9

Flagged orders per category, with the percentage each represents of that category's rows.

The rates differ by category, which is what you would expect when the
threshold is computed once across the whole column: categories with
inherently larger orders cross a fence set by the mixture.

That is an argument for computing fences **within group** when the groups
have genuinely different scales. A 5,000 Technology order and a 5,000
Office Supplies order are not equally unusual, and a single global fence
treats them as if they were.

In [ ]:
work = orders.copy()
q1, q3 = sales.quantile(0.25), sales.quantile(0.75)
iqr = q3 - q1
work["Outlier"] = (work["Sales"] < q1 - 1.5 * iqr) | (work["Sales"] > q3 + 1.5 * iqr)

by_cat = work.groupby("Category").agg(
    orders=("OrderID", "count"),
    flagged=("Outlier", "sum"),
)
by_cat["pct"] = (100 * by_cat["flagged"] / by_cat["orders"]).round(1)
print(by_cat.to_string())

### Question 10

z-score on a text column -> **raises** `TypeError: Cannot perform reduction 'mean' with string dtype`.

`Region` is `str`, and `.mean()` has nothing to compute.

A clean failure, and it is worth noticing that it comes from `.mean()`
rather than from anything outlier-specific. Neither the IQR rule nor the
z-score checks that its input is numeric — they are just arithmetic, and the
arithmetic is what objects.

That matters because a numeric column read as text — the `?`-contaminated
`Discount` from the week 3, day 1 class — would fail here in exactly the same way,
and the message would send you looking at your outlier code instead of at
your `read_csv` call.

In [ ]:
print("Region dtype:", orders["Region"].dtype)
print((orders["Region"] - orders["Region"].mean()) / orders["Region"].std())